# 从零实现 RealNVP Normalizing Flow：可逆耦合层、精确似然与采样

Normalizing flow 用一串可逆映射把数据 $x$ 变到简单 base distribution $z$。与 VAE 不同，RealNVP 在维度不变且映射可逆时能用变量替换公式计算精确 log-likelihood。本 Notebook 手写 affine coupling、正逆传播、log-determinant、训练、采样与制品发布，不调用现成 flow 包。

受控二维四峰数据用于验证机制。似然下降和样本落在数据范围只是 smoke test，不能替代高维图像/密度估计中的 bits-per-dimension、OOD 反常似然、生成质量或校准评估。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED53 = 5301  # 计算并保存当前步骤的中间状态。
random.seed(SEED53); np.random.seed(SEED53); torch.manual_seed(SEED53)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE53 = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

def canonical_json53(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 返回当前分支计算出的结果。

def sha53(value):  # 定义本节可复用的核心函数。
    return hashlib.sha256(value).hexdigest()  # 返回当前分支计算出的结果。

assert DEVICE53.type == "cpu" and torch.get_num_threads() == 1  # 用受控断言验证关键不变量。

## 1. 数据、切分与 train-only 标准化

四个 Gaussian 中心位于 `(-2,-2),(-2,2),(2,-2),(2,2)`，标准差 0.28。训练/验证/测试分别由独立 seed 生成，不能先合并再切分。normalizer 只由 train 推导：`x_norm=(x-mean_train)/std_train`。

张量合同是 `x:[B,2]`。二维数据便于画图，但这里不依赖可视化；数据 seed、中心、噪声、每个 split 和 normalizer 都进入发布 manifest。

In [ ]:
CENTERS53 = torch.tensor([[-2., -2.], [-2., 2.], [2., -2.], [2., 2.]])  # 计算并保存当前步骤的中间状态。

def make_mixture53(count, seed, noise=0.28):  # 定义本节可复用的核心函数。
    if count < 4 or noise <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("invalid_mixture_config")  # 遇到非法合同立即显式失败。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    labels = torch.randint(0, len(CENTERS53), (count,), generator=generator)  # 计算并保存当前步骤的中间状态。
    points = CENTERS53[labels] + noise * torch.randn(count, 2, generator=generator)  # 计算并保存当前步骤的中间状态。
    return points, labels  # 返回当前分支计算出的结果。

train_raw53, train_labels53 = make_mixture53(640, SEED53 + 1)  # 计算并保存当前步骤的中间状态。
val_raw53, val_labels53 = make_mixture53(160, SEED53 + 2)  # 计算并保存当前步骤的中间状态。
test_raw53, test_labels53 = make_mixture53(160, SEED53 + 3)  # 计算并保存当前步骤的中间状态。
mean53 = train_raw53.mean(0); std53 = train_raw53.std(0, unbiased=False).clamp_min(1e-6)  # 计算并保存当前步骤的中间状态。
normalize53 = lambda x: (x - mean53) / std53  # 计算并保存当前步骤的中间状态。
train53, val53, test53 = map(normalize53, (train_raw53, val_raw53, test_raw53))  # 计算并保存当前步骤的中间状态。
assert train53.shape == (640, 2) and val53.shape == test53.shape == (160, 2)  # 用受控断言验证关键不变量。
assert torch.allclose(train53.mean(0), torch.zeros(2), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(train53.std(0, unbiased=False), torch.ones(2), atol=1e-6)  # 用受控断言验证关键不变量。
regenerated53, _ = make_mixture53(640, SEED53 + 1)  # 计算并保存当前步骤的中间状态。
assert torch.equal(regenerated53, train_raw53)  # 用受控断言验证关键不变量。
assert not torch.equal(train_raw53[:160], val_raw53)  # 用受控断言验证关键不变量。

## 2. Affine coupling 的三角 Jacobian

二值 mask `m:[D]` 把输入分为保留部分 $x_a=m\odot x$ 与变换部分：

$$y_a=x_a,\quad y_b=(1-m)\odot[x\odot\exp(s(x_a))+t(x_a)].$$

因为 $s,t$ 只读取保留坐标，Jacobian 为三角结构，$\log|\det J|=\sum_{b}s_b$。为防 `exp` 溢出，把网络输出经 `tanh` 限制到 `[-max_log_scale,max_log_scale]`。mask 必须同时含 0 和 1；全保留或全变换都不是有效 coupling。

In [ ]:
class CouplingNet53(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.net = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 2 * dim))  # 计算并保存当前步骤的中间状态。
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.net(x)  # 返回当前分支计算出的结果。

class AffineCoupling53(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, mask, hidden_dim=32, max_log_scale=1.5):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        mask = torch.as_tensor(mask, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        if mask.ndim != 1 or not torch.all((mask == 0) | (mask == 1)) or mask.sum() in (0, mask.numel()):  # 按当前条件选择后续控制路径。
            raise ValueError("mask_must_mix_zero_and_one")  # 遇到非法合同立即显式失败。
        if not math.isfinite(max_log_scale) or not 0 < max_log_scale <= 5:  # 按当前条件选择后续控制路径。
            raise ValueError("max_log_scale_finite_range_contract")  # 遇到非法合同立即显式失败。
        self.register_buffer("mask", mask)  # 执行当前语句以推进本节示例。
        self.max_log_scale = float(max_log_scale)  # 计算并保存当前步骤的中间状态。
        self.conditioner = CouplingNet53(mask.numel(), hidden_dim)  # 计算并保存当前步骤的中间状态。

    def _coupling_parameters(self, kept):  # 定义本节可复用的核心函数。
        raw_scale, shift = self.conditioner(kept).chunk(2, dim=-1)  # 计算并保存当前步骤的中间状态。
        transformed = 1.0 - self.mask  # 计算并保存当前步骤的中间状态。
        scale = self.max_log_scale * torch.tanh(raw_scale) * transformed  # 计算并保存当前步骤的中间状态。
        shift = shift * transformed  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(scale).all() or not torch.isfinite(shift).all():  # 按当前条件选择后续控制路径。
            raise ValueError("nonfinite_coupling_parameters")  # 遇到非法合同立即显式失败。
        return scale, shift  # 返回当前分支计算出的结果。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 2 or x.shape[1] != self.mask.numel() or not torch.isfinite(x).all():  # 按当前条件选择后续控制路径。
            raise ValueError("coupling_input_contract")  # 遇到非法合同立即显式失败。
        kept = x * self.mask; scale, shift = self._coupling_parameters(kept)  # 计算并保存当前步骤的中间状态。
        y = kept + (1 - self.mask) * (x * torch.exp(scale) + shift)  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(y).all():  # 按当前条件选择后续控制路径。
            raise ValueError("nonfinite_coupling_output")  # 遇到非法合同立即显式失败。
        return y, scale.sum(-1)  # 返回当前分支计算出的结果。

    def inverse(self, y):  # 定义本节可复用的核心函数。
        if y.ndim != 2 or y.shape[1] != self.mask.numel() or not torch.isfinite(y).all():  # 按当前条件选择后续控制路径。
            raise ValueError("coupling_input_contract")  # 遇到非法合同立即显式失败。
        kept = y * self.mask; scale, shift = self._coupling_parameters(kept)  # 计算并保存当前步骤的中间状态。
        x = kept + (1 - self.mask) * ((y - shift) * torch.exp(-scale))  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(x).all():  # 按当前条件选择后续控制路径。
            raise ValueError("nonfinite_coupling_output")  # 遇到非法合同立即显式失败。
        return x, -scale.sum(-1)  # 返回当前分支计算出的结果。

coupling_probe53 = AffineCoupling53([1, 0], hidden_dim=8)  # 计算并保存当前步骤的中间状态。
x_probe53 = torch.tensor([[0.3, -0.7], [1.2, 0.4]])  # 计算并保存当前步骤的中间状态。
y_probe53, ld_probe53 = coupling_probe53(x_probe53)  # 计算并保存当前步骤的中间状态。
x_back53, ild_probe53 = coupling_probe53.inverse(y_probe53)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(x_probe53, x_back53, atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(ld_probe53 + ild_probe53, torch.zeros(2), atol=1e-7)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    AffineCoupling53([1, 1]); raise AssertionError("degenerate mask accepted")  # 执行当前语句以推进本节示例。
except ValueError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "mask_must_mix_zero_and_one"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    AffineCoupling53([1,0],max_log_scale=float("inf")); raise AssertionError("infinite log scale accepted")  # 计算并保存当前步骤的中间状态。
except ValueError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "max_log_scale_finite_range_contract"  # 用受控断言验证关键不变量。

## 3. RealNVP 正向、逆向与精确 log-probability

多层 coupling 交替 `[1,0]` 与 `[0,1]`，使两个维度都能被变换。正向 `x -> z` 累加每层 log-det；逆向按层倒序执行。标准正态 base 下：

$$\log p_X(x)=\log p_Z(f(x))+\log|\det J_f(x)|.$$

`forward(x)` 返回 `z:[B,2], log_det:[B]`；`inverse(z)` 返回重建与逆 log-det；`sample(n,generator)` 从显式随机流采样后逆变换。

In [ ]:
class StandardNormal53:  # 定义承载本节状态与行为的数据结构。
    @staticmethod  # 为下方定义附加声明式配置。
    def log_prob(z):  # 定义本节可复用的核心函数。
        if z.ndim != 2 or not torch.isfinite(z).all():  # 按当前条件选择后续控制路径。
            raise ValueError("base_input_contract")  # 遇到非法合同立即显式失败。
        return -0.5 * (z.square() + math.log(2 * math.pi)).sum(-1)  # 返回当前分支计算出的结果。

    @staticmethod  # 为下方定义附加声明式配置。
    def sample(count, dim, generator):  # 定义本节可复用的核心函数。
        if count < 1 or dim < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_sample_shape")  # 遇到非法合同立即显式失败。
        if not isinstance(generator, torch.Generator):  # 按当前条件选择后续控制路径。
            raise TypeError("explicit_torch_generator_required")  # 遇到非法合同立即显式失败。
        return torch.randn(count, dim, generator=generator)  # 返回当前分支计算出的结果。

class RealNVP53(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim=2, hidden_dim=32, num_layers=6, max_log_scale=1.5):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim != 2 or num_layers < 2:  # 按当前条件选择后续控制路径。
            raise ValueError("this_teaching_model_requires_dim2_and_two_layers")  # 遇到非法合同立即显式失败。
        masks = [[1, 0] if index % 2 == 0 else [0, 1] for index in range(num_layers)]  # 计算并保存当前步骤的中间状态。
        self.dim, self.hidden_dim, self.num_layers, self.max_log_scale = dim, hidden_dim, num_layers, max_log_scale  # 计算并保存当前步骤的中间状态。
        self.layers = nn.ModuleList([AffineCoupling53(mask, hidden_dim, max_log_scale) for mask in masks])  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        z = x; total = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)  # 计算并保存当前步骤的中间状态。
        for layer in self.layers:  # 遍历输入元素以累积或检查结果。
            z, log_det = layer(z); total = total + log_det  # 计算并保存当前步骤的中间状态。
        return z, total  # 返回当前分支计算出的结果。

    def inverse(self, z):  # 定义本节可复用的核心函数。
        x = z; total = torch.zeros(z.shape[0], dtype=z.dtype, device=z.device)  # 计算并保存当前步骤的中间状态。
        for layer in reversed(self.layers):  # 遍历输入元素以累积或检查结果。
            x, log_det = layer.inverse(x); total = total + log_det  # 计算并保存当前步骤的中间状态。
        return x, total  # 返回当前分支计算出的结果。

    def log_prob(self, x):  # 定义本节可复用的核心函数。
        z, log_det = self(x)  # 计算并保存当前步骤的中间状态。
        return StandardNormal53.log_prob(z) + log_det  # 返回当前分支计算出的结果。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def sample(self, count, generator):  # 定义本节可复用的核心函数。
        z = StandardNormal53.sample(count, self.dim, generator)  # 计算并保存当前步骤的中间状态。
        return self.inverse(z)[0]  # 返回当前分支计算出的结果。

model_probe53 = RealNVP53(hidden_dim=12, num_layers=4)  # 计算并保存当前步骤的中间状态。
z_probe53, total_ld53 = model_probe53(x_probe53)  # 计算并保存当前步骤的中间状态。
reconstruction53, inverse_ld53 = model_probe53.inverse(z_probe53)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(reconstruction53, x_probe53, atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(total_ld53 + inverse_ld53, torch.zeros(2), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(model_probe53.log_prob(x_probe53), StandardNormal53.log_prob(z_probe53) + total_ld53)  # 用受控断言验证关键不变量。
g1_53 = torch.Generator().manual_seed(3); g2_53 = torch.Generator().manual_seed(3)  # 计算并保存当前步骤的中间状态。
assert torch.equal(model_probe53.sample(5, g1_53), model_probe53.sample(5, g2_53))  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    model_probe53.sample(2,None); raise AssertionError("implicit global RNG accepted")  # 执行当前语句以推进本节示例。
except TypeError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "explicit_torch_generator_required"  # 用受控断言验证关键不变量。

## 4. 用 autograd Jacobian 交叉验证 log-det

“正反能还原”仍可能同时把 log-det 写错。二维 oracle 对单点计算完整 Jacobian `J:[2,2]`，用 `slogdet(J)` 与模型累加值比较。这个测试会抓到漏乘 mask、逆向符号错误和 batch 维误求和。

初始化时最后一层为零，coupling 接近恒等映射；为了让 oracle 非平凡，给 conditioner 最后一层写入固定 bias。

In [ ]:
jacobian_layer53 = AffineCoupling53([1, 0], hidden_dim=6, max_log_scale=1.5).double()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    jacobian_layer53.conditioner.net[-1].bias.copy_(torch.tensor([0.0, 0.2, 0.0, -0.15], dtype=torch.float64))  # 计算并保存当前步骤的中间状态。
point53 = torch.tensor([0.4, -0.2], dtype=torch.float64, requires_grad=True)  # 计算并保存当前步骤的中间状态。
def transform_single53(value):  # 定义本节可复用的核心函数。
    return jacobian_layer53(value[None, :])[0].squeeze(0)  # 返回当前分支计算出的结果。
jacobian53 = torch.autograd.functional.jacobian(transform_single53, point53)  # 计算并保存当前步骤的中间状态。
_, oracle_ld53 = jacobian_layer53(point53.detach()[None, :])  # 计算并保存当前步骤的中间状态。
sign53, slogdet53 = torch.linalg.slogdet(jacobian53)  # 计算并保存当前步骤的中间状态。
assert sign53 > 0 and torch.allclose(slogdet53, oracle_ld53.squeeze(0), atol=1e-10)  # 用受控断言验证关键不变量。
wrong_unscaled53 = jacobian_layer53.conditioner(point53.detach()[None, :] * jacobian_layer53.mask).chunk(2, -1)[0][0, 1]  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(slogdet53, wrong_unscaled53, atol=1e-5)  # 用受控断言验证关键不变量。
assert jacobian53.shape == (2, 2) and torch.isfinite(jacobian53).all()  # 用受控断言验证关键不变量。

## 5. 最大似然训练

训练最小化 `NLL=-mean(log_prob(x_train))`。每步用独立 generator 抽 minibatch；验证集只做选型，测试集最后一次报告。coupling 初始为恒等，初始 NLL 就是标准正态对标准化数据的拟合，后续下降说明 flow 学到了非 Gaussian 四峰结构。

这里只比较同一预处理、同一维度下的 NLL；不同 dequantization、normalizer 或数据支持集的似然不能直接横比。

In [ ]:
torch.manual_seed(SEED53)  # 执行当前语句以推进本节示例。
model53 = RealNVP53(hidden_dim=32, num_layers=6).to(DEVICE53)  # 计算并保存当前步骤的中间状态。
optimizer53 = torch.optim.Adam(model53.parameters(), lr=0.004)  # 计算并保存当前步骤的中间状态。
batch_generator53 = torch.Generator().manual_seed(SEED53 + 10)  # 计算并保存当前步骤的中间状态。
with torch.no_grad(): initial_nll53 = float(-model53.log_prob(val53).mean())  # 在受管理的上下文中执行操作。
history53 = []  # 计算并保存当前步骤的中间状态。
for step53 in range(320):  # 遍历输入元素以累积或检查结果。
    index53 = torch.randint(0, train53.shape[0], (128,), generator=batch_generator53)  # 计算并保存当前步骤的中间状态。
    loss53 = -model53.log_prob(train53[index53]).mean()  # 计算并保存当前步骤的中间状态。
    optimizer53.zero_grad(set_to_none=True); loss53.backward()  # 计算并保存当前步骤的中间状态。
    torch.nn.utils.clip_grad_norm_(model53.parameters(), 5.0); optimizer53.step()  # 执行当前语句以推进本节示例。
    if step53 % 40 == 0: history53.append(float(loss53.detach()))  # 按当前条件选择后续控制路径。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_val_nll53 = float(-model53.log_prob(val53).mean())  # 计算并保存当前步骤的中间状态。
    test_nll53 = float(-model53.log_prob(test53).mean())  # 计算并保存当前步骤的中间状态。
assert final_val_nll53 < initial_nll53 - 0.15  # 用受控断言验证关键不变量。
assert math.isfinite(test_nll53) and all(math.isfinite(value) for value in history53)  # 用受控断言验证关键不变量。
assert any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in model53.parameters())  # 用受控断言验证关键不变量。
print({"initial_val_nll": round(initial_nll53, 4), "final_val_nll": round(final_val_nll53, 4), "test_nll": round(test_nll53, 4)})  # 执行当前语句以推进本节示例。

## 6. 采样、支持集与 OOD 边界

采样路径是 `z~N(0,I) -> inverse(z) -> denormalize`。必须复用一个显式 generator，不能在循环或每个 batch 内重置 seed。由于可逆 flow 通常对整个 $\mathbb R^D$ 给正密度，“高似然”不自动等于“属于训练语义”；高维 flow 尤其可能给 OOD 更高似然。

这里检查样本有限、正逆一致以及多数样本靠近某个中心，只作为受控分布的 sanity check。

In [ ]:
sample_generator53 = torch.Generator().manual_seed(5319)  # 计算并保存当前步骤的中间状态。
samples_norm53 = model53.sample(400, sample_generator53)  # 计算并保存当前步骤的中间状态。
samples_raw53 = samples_norm53 * std53 + mean53  # 计算并保存当前步骤的中间状态。
distances53 = torch.cdist(samples_raw53, CENTERS53).min(-1).values  # 计算并保存当前步骤的中间状态。
assert samples_raw53.shape == (400, 2) and torch.isfinite(samples_raw53).all()  # 用受控断言验证关键不变量。
assert float((distances53 < 1.2).float().mean()) > 0.65  # 用受控断言验证关键不变量。
z_samples53, ld_samples53 = model53(samples_norm53[:20])  # 计算并保存当前步骤的中间状态。
recovered_samples53, ild_samples53 = model53.inverse(z_samples53)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(recovered_samples53, samples_norm53[:20], atol=2e-5)  # 用受控断言验证关键不变量。
assert torch.allclose(ld_samples53 + ild_samples53, torch.zeros(20), atol=2e-5)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    model53(torch.tensor([[float("inf"), 0.0]])); raise AssertionError("infinite sample accepted")  # 执行当前语句以推进本节示例。
except ValueError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "coupling_input_contract"  # 用受控断言验证关键不变量。

## 7. 标准化也会改变密度单位

模型学习的是标准化坐标 $u=(x-\mu)/\sigma$ 上的密度。若服务要返回原始坐标密度，还必须再应用一次变量替换：

$$\log p_X(x)=\log p_U((x-\mu)/\sigma)-\sum_d\log\sigma_d.$$

漏掉这一项不会影响同一 normalizer 下的排序，却会让 NLL、阈值和跨版本比较整体偏移。因此 mean/std 不只是前处理参数，也是密度语义的一部分。

In [ ]:
def raw_log_prob53(model, raw_points, mean, std):  # 定义本节可复用的核心函数。
    if raw_points.ndim != 2 or mean.shape != std.shape or mean.shape != (raw_points.shape[1],):  # 按当前条件选择后续控制路径。
        raise ValueError("raw_density_shape_contract")  # 遇到非法合同立即显式失败。
    if raw_points.dtype != mean.dtype or raw_points.dtype != std.dtype or raw_points.device != mean.device or raw_points.device != std.device:  # 按当前条件选择后续控制路径。
        raise ValueError("raw_density_dtype_device_contract")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(raw_points).all() or not torch.isfinite(mean).all() or not torch.isfinite(std).all() or not torch.all(std > 0):  # 按当前条件选择后续控制路径。
        raise ValueError("raw_density_finite_positive_contract")  # 遇到非法合同立即显式失败。
    normalized = (raw_points - mean) / std  # 计算并保存当前步骤的中间状态。
    return model.log_prob(normalized) - std.log().sum()  # 返回当前分支计算出的结果。

raw_lp53 = raw_log_prob53(model53, test_raw53[:6], mean53, std53)  # 计算并保存当前步骤的中间状态。
manual_raw_lp53 = model53.log_prob(test53[:6]) - std53.log().sum()  # 计算并保存当前步骤的中间状态。
assert torch.allclose(raw_lp53, manual_raw_lp53, atol=1e-7)  # 用受控断言验证关键不变量。
shifted_mean53 = mean53 + 0.5  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(raw_log_prob53(model53, test_raw53[:6], shifted_mean53, std53), raw_lp53)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    raw_log_prob53(model53, test_raw53[:2], mean53, torch.tensor([1.0, 0.0])); raise AssertionError("zero scale accepted")  # 执行当前语句以推进本节示例。
except ValueError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "raw_density_finite_positive_contract"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    raw_log_prob53(model53,test_raw53[:2],mean53,torch.tensor([float("inf"),1.])); raise AssertionError("infinite std accepted")  # 执行当前语句以推进本节示例。
except ValueError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "raw_density_finite_positive_contract"  # 用受控断言验证关键不变量。

## 8. 发布合同与整体重签攻击

manifest 绑定：模型层数/hidden/scale clamp、交替 mask、标准正态 base、四个中心与噪声、split seed/count/hash、train-only mean/std、训练步数与随机流。loader 重新生成三个 split 并重算 normalizer，而不是只比较调用方提供的统计量。

publisher registry 位于 package 外并只读；state 语义摘要包含每个 tensor 的 key、dtype、shape、bytes。loader 返回 `PublishedFlow53`，把 train-only mean/std、原单位 `raw_log_prob` 与 `sample_raw` 封装在一起，避免调用方忘记密度单位的 Jacobian。替换模型后重算内部所有摘要仍会得到新 bundle，因此被 registry 拒绝。

In [ ]:
def tensor_hash53(tensor):  # 定义本节可复用的核心函数。
    value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
    payload = str(value.dtype).encode() + canonical_json53(list(value.shape)).encode() + value.numpy().tobytes()  # 计算并保存当前步骤的中间状态。
    return sha53(payload)  # 返回当前分支计算出的结果。

def dataset_hash53(points, labels):  # 定义本节可复用的核心函数。
    return sha53((tensor_hash53(points) + tensor_hash53(labels)).encode())  # 返回当前分支计算出的结果。

def state_digest53(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key, tensor in sorted(state.items()):  # 遍历输入元素以累积或检查结果。
        digest.update(key.encode()); digest.update(tensor_hash53(tensor).encode())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

manifest53 = {  # 计算并保存当前步骤的中间状态。
    "artifact_id": "realnvp-four-mode-v1", "version": 1,  # 执行当前语句以推进本节示例。
    "model_config": {"dim": 2, "hidden_dim": 32, "num_layers": 6, "max_log_scale": 1.5},  # 执行当前语句以推进本节示例。
    "base": "standard_normal_2d", "masks": [[1, 0], [0, 1]] * 3,  # 执行当前语句以推进本节示例。
    "data": {"centers": CENTERS53.tolist(), "noise": 0.28,  # 执行当前语句以推进本节示例。
             "splits": {"train": [640, SEED53 + 1, dataset_hash53(train_raw53, train_labels53)],  # 执行当前语句以推进本节示例。
                        "val": [160, SEED53 + 2, dataset_hash53(val_raw53, val_labels53)],  # 执行当前语句以推进本节示例。
                        "test": [160, SEED53 + 3, dataset_hash53(test_raw53, test_labels53)]}},  # 执行当前语句以推进本节示例。
    "preprocess": {"kind": "train_only_standardize", "mean": mean53.tolist(), "std": std53.tolist()},  # 执行当前语句以推进本节示例。
    "training": {"seed": SEED53, "optimizer": "Adam", "steps": 320, "batch_size": 128, "lr": 0.004, "grad_clip_norm": 5.0},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

def build_package53(model, manifest):  # 定义本节可复用的核心函数。
    buffer = io.BytesIO(); torch.save(model.state_dict(), buffer); raw = buffer.getvalue()  # 计算并保存当前步骤的中间状态。
    state = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    manifest_sha = sha53(canonical_json53(manifest).encode()); semantic = state_digest53(state); raw_sha = sha53(raw)  # 计算并保存当前步骤的中间状态。
    bundle = sha53(canonical_json53({"manifest_sha": manifest_sha, "state_digest": semantic, "state_bytes_sha": raw_sha}).encode())  # 计算并保存当前步骤的中间状态。
    return {"manifest": copy.deepcopy(manifest), "manifest_sha": manifest_sha, "state_bytes": raw,  # 返回当前分支计算出的结果。
            "state_digest": semantic, "state_bytes_sha": raw_sha, "bundle_digest": bundle}  # 执行当前语句以推进本节示例。

package53 = build_package53(model53, manifest53)  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY53 = MappingProxyType({("realnvp-four-mode-v1", 1): package53["bundle_digest"]})  # 计算并保存当前步骤的中间状态。

class PublishedFlow53:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, model, mean, std):  # 定义本节可复用的核心函数。
        self._model = model  # 计算并保存当前步骤的中间状态。
        self._mean = mean.detach().clone()  # 计算并保存当前步骤的中间状态。
        self._std = std.detach().clone()  # 计算并保存当前步骤的中间状态。
    @property  # 为下方定义附加声明式配置。
    def mean(self): return self._mean.clone()  # 定义本节可复用的核心函数。
    @property  # 为下方定义附加声明式配置。
    def std(self): return self._std.clone()  # 定义本节可复用的核心函数。
    def raw_log_prob(self, raw_points): return raw_log_prob53(self._model,raw_points,self._mean,self._std)  # 定义本节可复用的核心函数。
    @torch.no_grad()  # 为下方定义附加声明式配置。
    def sample_raw(self,count,generator): return self._model.sample(count,generator)*self._std+self._mean  # 定义本节可复用的核心函数。

def load_flow53(package):  # 定义本节可复用的核心函数。
    manifest = package["manifest"]; key = (manifest.get("artifact_id"), manifest.get("version"))  # 计算并保存当前步骤的中间状态。
    if PUBLISHER_REGISTRY53.get(key) != package.get("bundle_digest"):  # 按当前条件选择后续控制路径。
        raise RuntimeError("publisher_registry_rejected_bundle")  # 遇到非法合同立即显式失败。
    if manifest != manifest53 or sha53(canonical_json53(manifest).encode()) != package["manifest_sha"]:  # 按当前条件选择后续控制路径。
        raise RuntimeError("manifest_contract_mismatch")  # 遇到非法合同立即显式失败。
    regenerated = [make_mixture53(count, seed) for count, seed, _ in manifest["data"]["splits"].values()]  # 计算并保存当前步骤的中间状态。
    for (points, labels), (_, _, expected) in zip(regenerated, manifest["data"]["splits"].values()):  # 遍历输入元素以累积或检查结果。
        if dataset_hash53(points, labels) != expected: raise RuntimeError("split_snapshot_mismatch")  # 按当前条件选择后续控制路径。
    train_points = regenerated[0][0]; derived_mean = train_points.mean(0); derived_std = train_points.std(0, unbiased=False).clamp_min(1e-6)  # 计算并保存当前步骤的中间状态。
    if not torch.allclose(derived_mean, torch.tensor(manifest["preprocess"]["mean"])) or not torch.allclose(derived_std, torch.tensor(manifest["preprocess"]["std"])):  # 按当前条件选择后续控制路径。
        raise RuntimeError("preprocess_not_derived_from_train")  # 遇到非法合同立即显式失败。
    if sha53(package["state_bytes"]) != package["state_bytes_sha"]: raise RuntimeError("state_bytes_mismatch")  # 按当前条件选择后续控制路径。
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    if state_digest53(state) != package["state_digest"]: raise RuntimeError("state_semantic_mismatch")  # 按当前条件选择后续控制路径。
    expected_bundle = sha53(canonical_json53({"manifest_sha": package["manifest_sha"], "state_digest": package["state_digest"], "state_bytes_sha": package["state_bytes_sha"]}).encode())  # 计算并保存当前步骤的中间状态。
    if expected_bundle != package["bundle_digest"]: raise RuntimeError("bundle_digest_mismatch")  # 按当前条件选择后续控制路径。
    restored = RealNVP53(**manifest["model_config"]); restored.load_state_dict(state); restored.eval()  # 计算并保存当前步骤的中间状态。
    return PublishedFlow53(restored,derived_mean,derived_std)  # 返回当前分支计算出的结果。

restored53 = load_flow53(package53)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(restored53.raw_log_prob(test_raw53[:8]), raw_log_prob53(model53,test_raw53[:8],mean53,std53), atol=1e-7)  # 用受控断言验证关键不变量。
published_samples53 = restored53.sample_raw(4,torch.Generator().manual_seed(88))  # 计算并保存当前步骤的中间状态。
direct_samples53 = model53.sample(4,torch.Generator().manual_seed(88))*std53+mean53  # 计算并保存当前步骤的中间状态。
assert torch.equal(published_samples53,direct_samples53)  # 用受控断言验证关键不变量。
leaked_mean53 = restored53.mean; leaked_mean53.add_(100)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(restored53.mean,mean53)  # 用受控断言验证关键不变量。
forged53 = RealNVP53(hidden_dim=32, num_layers=6)  # 计算并保存当前步骤的中间状态。
forged_package53 = build_package53(forged53, manifest53)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_flow53(forged_package53); raise AssertionError("re-signed forged flow accepted")  # 执行当前语句以推进本节示例。
except RuntimeError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "publisher_registry_rejected_bundle"  # 用受控断言验证关键不变量。
assert isinstance(PUBLISHER_REGISTRY53, MappingProxyType)  # 用受控断言验证关键不变量。

## 9. 复杂度、失败模式与原始来源

每层 conditioner 的成本约为 MLP 成本，训练必须同时跑全部 coupling；采样走逆序但无需迭代数百个 diffusion step。常见错误是漏加 log-det、逆变换符号错误、mask 不交替、scale 无界、用全数据标准化、在离散像素上忘记 dequantization，以及用 likelihood 单独判断 OOD。

- Dinh, Sohl-Dickstein & Bengio, [Density Estimation using Real NVP](https://arxiv.org/abs/1605.08803), ICLR 2017。
- Rezende & Mohamed, [Variational Inference with Normalizing Flows](https://arxiv.org/abs/1505.05770), ICML 2015。
- Papamakarios et al., [Normalizing Flows for Probabilistic Modeling and Inference](https://arxiv.org/abs/1912.02762), JMLR 2021。

本例只复现二维 affine coupling 与精确变量替换，没有覆盖图像多尺度 squeeze、invertible 1x1 convolution 或高维生产训练。